# **BINARY CLASSIFICATION SETUP**

In [ ]:
"""IMPORTS"""

import shutil
from pathlib import Path
import importlib
import sys
import time
import warnings
from collections import Counter
import contextlib, io

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support



import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "classification":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    

HU_HFSTOL = PROJECT_ROOT / "input_files" / "hu.hfstol"

HFST_LOOKUP_PATH = os.environ.get("HFST_LOOKUP")

if HFST_LOOKUP_PATH is None:
    hfst_from_path = shutil.which("hfst-lookup.exe") or shutil.which("hfst-lookup")
    if hfst_from_path:
        HFST_LOOKUP_PATH = hfst_from_path
    else:
        candidate = Path.home() / "hfst" / "bin" / "hfst-lookup.exe"
        if candidate.exists():
            HFST_LOOKUP_PATH = str(candidate)

if HFST_LOOKUP_PATH is None:
    raise FileNotFoundError(
        "hfst-lookup was not found. Set HFST_LOOKUP before running this notebook, "
        "or add the HFST bin folder to PATH."
    )

os.environ["HFST_LOOKUP"] = str(HFST_LOOKUP_PATH)
os.environ["HU_HFSTOL"] = str(HU_HFSTOL)


from helpers import classification
from helpers import emmorph_pos
from helpers import etymology_analysis_tables as eat
from helpers import etymology_filters as ety_filters
from helpers import ipa_enrichment
from helpers import phonology
from helpers import semantic_features
from helpers import ueszweb_abbreviations as ab

for module in (
    ety_filters,
    emmorph_pos,
    ipa_enrichment,
    phonology,
    classification,
    eat,
    semantic_features,
):
    importlib.reload(module)

from helpers.phonology import (
    V_PREFIXES,
    yes_no_to_bool,
    tokenize_ipa_transcribed_word,
    phoneme_ngrams,
    collect_ngrams_multi,
    VOWEL_CLASS,
    VOWEL_BASES,
    vowel_string,
    vowel_pattern_tokens,
    extract_vowel_phonemes,
    vowel_string_token,
    make_ngram_documents,
    filter_real_entries,
    add_custom_labels,
    build_binary_loan_dataset,
    #build_family_dataset_multiclass, --> moved to classification.py
)

from helpers.classification import (
    CERTAINTY_TO_WEIGHT,
    extract_vowel_coefs,
    summarize_coef_maps,
    balance_indices_equalize,
    train_test_loan_classifier,
    train_test_family_classifier,
)

from helpers.emmorph_pos import (
    SEGMENT_RE,
    emmorph_analyse_many,
)

from helpers.ipa_enrichment import (
    phonemize_lemmas,
)


from helpers.etymology_analysis_tables import (
    ExperimentConfig,
    DEFAULT_EXPERIMENT_CONFIG,
    prepare_for_tables,
    make_count_table,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("Project root:", PROJECT_ROOT)
print("Using classification module:", classification.__file__)
print("HFST_LOOKUP:", os.environ.get("HFST_LOOKUP", "not set"))
print("HU_HFSTOL:", os.environ.get("HU_HFSTOL", "not set"))

In [ ]:
INPUT_CSV = PROJECT_ROOT / "input_files" / "etymology_database_gold_thesis.csv"

df_raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", dtype=str).fillna("") # --> dataframe aobut the enriched csv file

print("Input CSV:", INPUT_CSV)
print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))


In [ ]:
importlib.reload(eat)

family_table, fig, ax = eat.plot_origin_by_period(
    df_raw,
    origin_types=("loan", "inherited"),
    formation_type="simplex",
    target_families=("Uralic", "Slavic", "German", "Turkic", "Latin", "Romance", "English"),
    save_path="origin_by_period.jpeg",
)

display(family_table)
plt.show()

In [ ]:
importlib.reload(eat)
ending_rates = eat.verb_ending_origin_table(df_raw)

display(ending_rates.round(1))

fig, ax = eat.plot_verb_ending_origin_rates(ending_rates)
plt.show()

fig.savefig("verb_endings_by_origin.jpeg", bbox_inches="tight")


In [ ]:
RANDOM_STATE = 23

BINARY_BASE_CONFIG = dict(
    ns=(2, 3),
    exclude_old_words=False,
    exclude_international_wanderword=False,
    add_vowel_string=False,
    add_vowel_patterns=False,
    include_missing_pos=True,
    strip_final_ik=False, # --> NOT USED
    min_df_grid=(2, 3),
    mode="kfold",
    equalize_classes=False,
    equalize_method="oversample",   # "undersample" | "oversample" --> ignored when equalize_classes=False
    k=5,
    random_state=RANDOM_STATE,
    pos_filter=None,
)



ABLATION_VARIANTS = {
    "ngrams": {"add_vowel_string": False, "add_vowel_patterns": False, "add_vowel_length_string": False, "articulatory_backend": None, "add_pos_features": False, "semantic_feature": None},
    "+vowel": {"add_vowel_string": True, "add_vowel_patterns": True, "add_vowel_length_string": True, "articulatory_backend": None, "add_pos_features": False, "semantic_feature": None},
    "+art": {"add_vowel_string": True, "add_vowel_patterns": True, "add_vowel_length_string": True, "articulatory_backend": "panphon", "add_pos_features": False, "semantic_feature": None},
    "+pos": {"add_vowel_string": True, "add_vowel_patterns": True, "add_vowel_length_string": True, "articulatory_backend": "panphon", "add_pos_features": True, "semantic_feature": None},
    "+sem": {"add_vowel_string": True, "add_vowel_patterns": True, "add_vowel_length_string": True, "articulatory_backend": "panphon", "add_pos_features": True, "semantic_feature": "odenet"},
}

# '+vowel' features are split into:
# add_vowel_patterns: count, harmony, vowel-class switches, rounding
# add_vowel_string: exact vowel sequence without length
# add_vowel_length_string: exact vowel sequence with length


binary_variant_order = ["ngrams", "+vowel", "+art", "+pos", "+sem"]
binary_variant_eval_tables = {}
binary_variant_summary_rows = []
binary_variant_models = {}

for name, overrides in ABLATION_VARIANTS.items():
    cfg = {**BINARY_BASE_CONFIG, **overrides}
    print(f"Running {name} ...")

    clf, vec, df_eval, best_min_df, summary = classification.train_test_loan_classifier(
        df_raw,
        **cfg,
        verbose=True,
    )

    binary_variant_eval_tables[name] = df_eval
    binary_variant_models[name] = {"clf": clf, "vec": vec, "summary": summary, "best_min_df": best_min_df, "cfg": cfg}

    binary_variant_summary_rows.append(pd.Series({
        "variant": name,
        "accuracy": summary["accuracy_mean"],
        "precision_macro": summary["precision_macro_mean"],
        "recall_macro": summary["recall_macro_mean"],
        "f1_macro": summary["f1_macro_mean"],
        "log_loss": summary["log_loss_mean"],
        "precision_native": summary["precision_native_mean"],
        "recall_native": summary["recall_native_mean"],
        "f1_native": summary["f1_native_mean"],
        "precision_loan": summary["precision_loan_mean"],
        "recall_loan": summary["recall_loan_mean"],
        "f1_loan": summary["f1_loan_mean"],
    }))

overall_results = (
    pd.DataFrame(binary_variant_summary_rows)
    .set_index("variant")
    [[
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "log_loss",
        "precision_native",
        "recall_native",
        "f1_native",
        "precision_loan",
        "recall_loan",
        "f1_loan",
    ]]
    .reindex(binary_variant_order)
)

df_bin = phonology.build_binary_loan_dataset(
    df_raw,
    exclude_old_words=BINARY_BASE_CONFIG["exclude_old_words"],
    exclude_international_wanderword=BINARY_BASE_CONFIG["exclude_international_wanderword"],
)

y = df_bin["y_loan"].astype(int).to_numpy()
skf = StratifiedKFold(n_splits=BINARY_BASE_CONFIG["k"], shuffle=True, random_state=RANDOM_STATE)

baseline_rows = []
for train_idx, test_idx in skf.split(np.zeros(len(y)), y):
    majority = pd.Series(y[train_idx]).mode().iloc[0]
    y_test = y[test_idx]
    y_pred = np.full_like(y_test, majority)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    p_cls, r_cls, f_cls, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1], average=None, zero_division=0)

    baseline_rows.append({
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f_macro,
        "log_loss": np.nan,
        "precision_native": p_cls[0],
        "recall_native": r_cls[0],
        "f1_native": f_cls[0],
        "precision_loan": p_cls[1],
        "recall_loan": r_cls[1],
        "f1_loan": f_cls[1],
    })

baseline_row = pd.DataFrame(baseline_rows).mean()
baseline_row.name = "majority"

overall_results = pd.concat([
    baseline_row.to_frame().T,
    overall_results,
])

overall_results["delta_f1_macro"] = overall_results["f1_macro"].diff()
overall_results["delta_log_loss"] = overall_results["log_loss"].diff()

display(overall_results.round(4))

# **TABLES FOR STATISTICS ABOUT THE BINARY CLASSIFICATION**

In [ ]:
""" CONFUSION MATRIX WITH FALSE POS AND FALSE NEG """

df_all_folds_sem = pd.concat(
    binary_variant_models["+sem"]["summary"]["fold_evals"],
    ignore_index=True,
)

conf_matrix_all_sem = confusion_matrix(
    df_all_folds_sem["y_true"],
    df_all_folds_sem["y_pred"],
    labels=[0, 1],
)

conf_matrix_all_sem = pd.DataFrame(
    conf_matrix_all_sem,
    index=["true non-loan", "true loan"],
    columns=["pred non-loan", "pred loan"],
)

display(conf_matrix_all_sem)

In [ ]:
for col in df_raw.columns:
    print(col)


In [ ]:
"""internal non-loan recall by derivation_base_origin"""

from helpers import ueszweb_abbreviations as ab

DONOR_FAMILY_NAMES = {
    family.strip().casefold()
    for family in ab.LANGUAGE_FAMILIES.keys()
}


def split_pipe_labels(x):
    if pd.isna(x):
        return []
    return [
        p.strip().casefold()
        for p in str(x).split("|")
        if p.strip()
    ]


def clean_base_origin_group(x):
    parts = split_pipe_labels(x)

    if not parts:
        return "missing"

    real_parts = [
        p for p in parts
        if p not in {"none", "nan", "missing", ""}
    ]

    if not real_parts:
        return "missing"

    # Any established donor-family label except Uralic is treated as loan-origin.
    if any((p in DONOR_FAMILY_NAMES and p != "uralic") for p in real_parts):
        return "loan-origin base"

    if "uralic" in real_parts:
        return "Uralic base"

    if "internal" in real_parts:
        return "internal base"

    if "onomatopoetic" in real_parts:
        return "onomatopoetic base"

    if "fictional" in real_parts:
        return "fictional base"

    if "unknown" in real_parts:
        return "unknown base"

    return "|".join(real_parts)


def ensure_eval_meta(df_eval, source_df, needed_cols):
    df = df_eval.copy()

    for col in needed_cols:
        if col not in df.columns:
            if col not in source_df.columns:
                raise KeyError(f"Column {col!r} is missing from both df_eval and source_df.")
            df[col] = source_df.loc[df.index, col]

    return df


def internal_base_origin_recall_collapsed(
    variant_models,
    source_df,
    *,
    variant_name="+sem",
    min_n_total=10,
):
    fold_evals = variant_models[variant_name]["summary"]["fold_evals"]
    all_rows = []

    for fold_id, df_eval in enumerate(fold_evals, start=1):
        df = ensure_eval_meta(
            df_eval,
            source_df,
            ["origin_type", "derivation_base_origin", "y_true", "y_pred"],
        ).copy()

        df["fold"] = fold_id

        # Only true internal non-loan forms
        df = df[
            (df["y_true"] == 0)
            & df["origin_type"].apply(lambda x: "internal" in split_pipe_labels(x))
        ].copy()

        df["base_origin_group"] = df["derivation_base_origin"].apply(clean_base_origin_group)

        all_rows.append(df)

    all_df = pd.concat(all_rows, ignore_index=True)

    out = (
        all_df
        .groupby("base_origin_group")
        .apply(lambda g: pd.Series({
            "n_total": int(len(g)),
            "recall_nonloan": float((g["y_pred"] == 0).mean()),
        }))
        .query("n_total >= @min_n_total")
        .sort_values("recall_nonloan", ascending=False)
    )

    out["n_total"] = out["n_total"].astype(int)

    return out, all_df


internal_base_origin_stats, internal_base_origin_rows = internal_base_origin_recall_collapsed(
    binary_variant_models,
    df_raw,
    variant_name="+sem",
    min_n_total=10,
)

display(internal_base_origin_stats.round(3))


In [ ]:
""" TOP 20 coefficients for non-loan and loan prediction """

TOP_N = 20
table_variant_order = ["ngrams", "+vowel", "+art", "+pos", "+sem"]


def get_feature_table(model_bundle):
    clf = model_bundle["clf"]
    vec = model_bundle["vec"]
    cfg = model_bundle["cfg"]
    summary = model_bundle["summary"]

    feature_names = list(vec.get_feature_names_out())

    backend = cfg.get("articulatory_backend")
    if backend is not None:
        feature_names.extend(classification.get_articulatory_feature_names(backend))
        feature_names.extend(classification.WORD_SCALAR_NAMES)

    feature_names.extend(summary.get("semantic_feature_names", []))
    feature_names.extend(summary.get("pos_feature_names", []))

    coefs = clf.coef_[0]

    if len(feature_names) != len(coefs):
        raise ValueError(
            f"Feature-name/coefficient mismatch: "
            f"{len(feature_names)} names but {len(coefs)} coefficients."
        )

    return pd.DataFrame({"feature": feature_names, "coef": coefs})


def top_feature_dataframe(direction="loan", top_n=20):
    assert direction in {"loan", "nonloan"}

    rows = []

    for variant_name in table_variant_order:
        feat_df = get_feature_table(binary_variant_models[variant_name])

        if direction == "loan":
            top = feat_df.sort_values("coef", ascending=False).head(top_n)
        else:
            top = feat_df.sort_values("coef", ascending=True).head(top_n)

        for rank, (_, row) in enumerate(top.iterrows(), start=1):
            rows.append({
                "direction": direction,
                "variant": variant_name,
                "rank": rank,
                "feature": row["feature"],
                "coef": row["coef"],
            })

    return pd.DataFrame(rows)


top_loan_df = top_feature_dataframe("loan", TOP_N)
top_nonloan_df = top_feature_dataframe("nonloan", TOP_N)

print(f"TOP {TOP_N} COEFFICIENTS FOR LOAN DIRECTION")
display(top_loan_df.head(TOP_N))
print(f"TOP {TOP_N} COEFFICIENTS FOR NON-LOAN DIRECTION")
display(top_nonloan_df.head(TOP_N))



In [ ]:
""" for POS-grouping and time period grouping """
def add_analysis_columns(df):
    df = df.reset_index(drop=True).copy()

    def make_pos_group(pos):
        parts = {p.strip().upper() for p in str(pos).split(";") if p.strip()}

        if "NOUN" in parts:
            return "noun"
        if "VERB" in parts:
            return "verb"
        if "ADJ" in parts or "ADV" in parts:
            return "adj_adv"

        return "other"

    df["pos_group"] = df["pos"].apply(make_pos_group)

    first_year_numeric = pd.to_numeric(df["first_year"], errors="coerce")

    df["first_year_bin"] = pd.cut(
        first_year_numeric,
        bins=eat.DEFAULT_PERIOD_BINS,
        labels=eat.DEFAULT_PERIOD_LABELS,
        right=True,
    ).astype("object")

    df["first_year_bin"] = df["first_year_bin"].fillna("missing year")

    return df


def _score_one_group(g, target_class, metric):
    target_label = {
        "native": 0,
        "nonloan": 0,
        "loan": 1,
    }[target_class]

    precision, recall, f1, _ = precision_recall_fscore_support(
        g["y_true"],
        g["y_pred"],
        labels=[target_label],
        average=None,
        zero_division=0,
    )

    return {
        "precision": precision[0],
        "recall": recall[0],
        "f1": f1[0],
    }[metric]


def build_metric_dataframe(
    bvariant_models,
    source_df,
    group_col,
    target_class,
    metric,
    *,
    variant_name="+sem",
    min_n=10,
    row_order=None,
    group_header="Group"):
    rows = []

    if group_col not in source_df.columns:
        raise KeyError(f"{group_col!r} is missing from source_df.")

    target_label = {
        "native": 0, # in case nonloan is not in there, probably not used
        "nonloan": 0,
        "loan": 1,
    }[target_class]

    fold_evals = bvariant_models[variant_name]["summary"]["fold_evals"]

    for fold_id, df_eval in enumerate(fold_evals, start=1):
        df = df_eval.copy()

        if group_col not in df.columns:
            df[group_col] = source_df.reindex(df.index)[group_col].values

        if group_col == "pos_group":
            df = df[df["pos_group"].isin(["verb", "noun", "adj_adv"])].copy()

        for group, g in df.groupby(group_col, dropna=False):
            if len(g) < min_n:
                continue

            rows.append({
                "fold": fold_id,
                group_header: group,
                "value": _score_one_group(g, target_class, metric),
                "n": int((g["y_true"] == target_label).sum()),
            })

    fold_df = pd.DataFrame(rows)

    out = (
        fold_df
        .groupby(group_header, as_index=False)
        .agg(
            value=("value", "mean"),
            std=("value", "std"),
            n=("n", "mean"),
        )
    )

    out["n"] = out["n"].round().astype(int)

    if row_order is not None:
        out = out[out[group_header].isin(row_order)].copy()

        out[group_header] = pd.Categorical(
            out[group_header],
            categories=row_order,
            ordered=True,
        )

        out = out.sort_values(group_header)

    out[variant_name] = out.apply(
        lambda row: f"{row['value']:.2f} ±{row['std']:.2f} (n={row['n']})",
        axis=1,
    )

    return out[[group_header, variant_name]].reset_index(drop=True), fold_df

""" for grouping """
df_raw = add_analysis_columns(df_raw)

period_order = list(eat.DEFAULT_PERIOD_LABELS) + ["missing year"]

pos_order = ["verb", "noun", "adj_adv"]

origin_order = [
    "internal",
    "fictional",
    "onomatopoetic",
    "inherited",
]

donor_family_order = [
    "Latin",
    "German",
    "international",
    "Slavic",
    "Romance",
    "Turkic",
    "wanderword",
    "English",
    "Indo-Iranian",
]


nonloan_f1_pos, nonloan_f1_pos_folds = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="pos_group",
    target_class="native",
    metric="f1",
    variant_name="+art",
    row_order=pos_order,
    group_header="POS",
)

loan_f1_pos, loan_f1_pos_folds = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="pos_group",
    target_class="loan",
    metric="f1",
    variant_name="+art",
    row_order=pos_order,
    group_header="POS",
)

nonloan_recall_origin, nonloan_recall_origin_folds = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="origin_type",
    target_class="native",
    metric="recall",
    variant_name="+sem",
    row_order=origin_order,
    group_header="Origin type",
)

loan_recall_donor, loan_recall_donor_folds = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="primary_donor_family",
    target_class="loan",
    metric="recall",
    variant_name="+sem",
    row_order=donor_family_order,
    group_header="Donor family",
)

nonloan_f1_period, nonloan_f1_period_folds = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="first_year_bin",
    target_class="native",
    metric="f1",
    variant_name="+sem",
    row_order=period_order,
    group_header="Period",
)

loan_f1_period, loan_f1_period_folds = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="first_year_bin",
    target_class="loan",
    metric="f1",
    variant_name="+sem",
    row_order=period_order,
    group_header="Period",
)


print("Non-loan F1 by POS group")
display(nonloan_f1_pos)

print("Loanword F1 by POS group")
display(loan_f1_pos)

print("Non-loan recall by origin type")
display(nonloan_recall_origin)

print("Loanword recall by donor family")
display(loan_recall_donor)

print("Non-loan F1 by attestation period")
display(nonloan_f1_period)

print("Loanword F1 by attestation period")
display(loan_f1_period)


In [ ]:
def assign_formation_group(formation_type):
    parts = {
        p.strip()
        for p in str(formation_type).split("|")
        if p.strip()
    }

    if parts == {"simplex"}:
        return "simplex"

    if parts == {"derivation"}:
        return "derivation"

    return "other"


def vowel_count_group_from_ipa(ipa):
    tokens = phonology.vowel_pattern_tokens(str(ipa or ""))

    vcount_token = next(
        (t for t in tokens if t.startswith("VCOUNT_")),
        None,
    )

    if vcount_token is None:
        return "missing"

    value = vcount_token.replace("VCOUNT_", "").lower()

    if value == "4plus":
        return "4+ vowels"

    if value == "1":
        return "1 vowel"

    return f"{value} vowels"


df_raw["formation_group"] = df_raw["formation_type"].apply(assign_formation_group)
df_raw["vowel_count_group"] = df_raw["ipa"].apply(vowel_count_group_from_ipa)

formation_order = ["simplex", "derivation", "other"]
vowel_count_order = ["0 vowels", "1 vowel", "2 vowels", "3 vowels", "4+ vowels", "missing"]


formation_nonloan_f1, _ = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="formation_group",
    target_class="native",
    metric="f1",
    variant_name="+sem",
    row_order=formation_order,
    group_header="Formation type",
)

formation_loan_f1, _ = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="formation_group",
    target_class="loan",
    metric="f1",
    variant_name="+sem",
    row_order=formation_order,
    group_header="Formation type",
)

vowel_nonloan_f1, _ = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="vowel_count_group",
    target_class="native",
    metric="f1",
    variant_name="+sem",
    row_order=vowel_count_order,
    group_header="Vowels",
)

vowel_loan_f1, _ = build_metric_dataframe(
    binary_variant_models,
    df_raw,
    group_col="vowel_count_group",
    target_class="loan",
    metric="f1",
    variant_name="+sem",
    row_order=vowel_count_order,
    group_header="Vowels",
)


print("Non-loan F1 by formation group")
display(formation_nonloan_f1)

print("Loanword F1 by formation group")
display(formation_loan_f1)

print("Non-loan F1 by vowel-count group")
display(vowel_nonloan_f1)

print("Loanword F1 by vowel-count group")
display(vowel_loan_f1)



# **MULTINOMIAL LOGISTIC REGRESSION**

In [ ]:

RANDOM_STATE = 23

# Reuse the repository-local input CSV loaded near the top of the notebook.
df_raw_multi = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", dtype=str).fillna("")


ABLATION_VARIANTS = {
    "ngrams": {"add_vowel_string": False, "add_vowel_patterns": False, "add_vowel_length_string": False, "articulatory_backend": None, "add_pos_features": False, "semantic_feature": None},
    "+vowel": {"add_vowel_string": True, "add_vowel_patterns": True, "add_vowel_length_string": True, "articulatory_backend": None, "add_pos_features": False, "semantic_feature": None},
    "+art": {"add_vowel_string": True, "add_vowel_patterns": True, "add_vowel_length_string": True, "articulatory_backend": "panphon", "add_pos_features": False, "semantic_feature": None},
    "+pos": {"add_vowel_string": True, "add_vowel_patterns": True, "add_vowel_length_string": True, "articulatory_backend": "panphon", "add_pos_features": True, "semantic_feature": None},
    "+sem": {"add_vowel_string": True, "add_vowel_patterns": True, "add_vowel_length_string": True, "articulatory_backend": "panphon", "add_pos_features": True, "semantic_feature": "odenet"},
}

multi_variant_order = ["ngrams", "+vowel", "+art", "+pos", "+sem"]


TARGET_FAMILIES = [
    "non-loan",
    "international/wanderword",
    "Latin",
    "German",
    "Romance",
    "English",
    "Slavic",
    "Turkic",
    "Other loan",
]


BASE_CONFIG = dict(
    target_families=TARGET_FAMILIES,
    ns=(2, 3),
    exclude_old_words=False,
    add_vowel_string=False,
    add_vowel_patterns=False,
    add_vowel_length_string=False,
    include_missing_pos=True, # ALWAYS TRUE
    strip_final_ik=False, # NOT USED
    min_df_grid=(2, 3),
    mode="kfold",
    equalize_classes=False,
    equalize_method="oversample", # ignored when equalize_class=False, it is NOT USED
    k=5,
    random_state=RANDOM_STATE,
    pos_filter=None,
)


multi_variant_eval_tables = {}
multi_variant_models = {}
multi_variant_summary_rows = []

# Sanity check: this prints the nine-class setup
df_multi_check = classification.build_origin_source_dataset_multiclass(
    df_raw_multi,
    target_families=TARGET_FAMILIES,
    exclude_old_words=False,
)

display(df_multi_check["family_clean"].value_counts())


for name, overrides in ABLATION_VARIANTS.items():
    cfg = {**BASE_CONFIG, **overrides}
    print(f"Running {name} ...")

    clf, vec, df_eval, best_min_df, summary = classification.train_test_family_classifier(
        df_raw_multi,   # <-- not df_multi
        **cfg,
        verbose=True,
    )

    multi_variant_eval_tables[name] = df_eval
    multi_variant_models[name] = {
        "clf": clf,
        "vec": vec,
        "summary": summary,
        "best_min_df": best_min_df,
        "cfg": cfg,
    }

    row = pd.Series({
        "variant": name,
        "best_min_df": best_min_df,
        "accuracy": summary["accuracy_mean"],
        "precision_macro": summary["precision_macro_mean"],
        "recall_macro": summary["recall_macro_mean"],
        "f1_macro": summary["f1_macro_mean"],
        "log_loss": summary["log_loss_mean"],
        "macro_f1_std": summary["macro_f1_std"],
    })

    multi_variant_summary_rows.append(row)

overall_results = (
    pd.DataFrame(multi_variant_summary_rows)
    .set_index("variant")
    [[
        "best_min_df",
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "log_loss",
        "macro_f1_std",
    ]]
    .reindex(multi_variant_order)
)

baseline_folds, baseline_summary = classification.majority_baseline_from_fold_evals(
    multi_variant_models[multi_variant_order[0]]["summary"]["fold_evals"]
)

baseline_row = pd.Series({
    "variant": "majority",
    "best_min_df": np.nan,
    "accuracy": baseline_summary["accuracy"],
    "precision_macro": baseline_summary["precision_macro"],
    "recall_macro": baseline_summary["recall_macro"],
    "f1_macro": baseline_summary["f1_macro"],
    "log_loss": np.nan,
    "macro_f1_std": baseline_summary["macro_f1_std"],
})

overall_results = pd.concat([
    baseline_row.to_frame().T.set_index("variant"),
    overall_results,
])


display(overall_results.round(1))



In [ ]:
import pandas as pd

def inspect_multiclass_eval(df_eval_multi, sample_n=60, random_state=42):
    # Confusion matrix: rows=true, cols=predicted
    print("CONFUSION MATRIX (rows=true, cols=predicted)")
    conf = pd.crosstab(
        df_eval_multi["true_family"],
        df_eval_multi["pred_family"],
        rownames=["True"],
        colnames=["Predicted"],
    )
    display(conf)

    # Row-normalised confusion matrix
    print("CONFUSION MATRIX (row-normalised, % of true class)")
    conf_pct = conf.div(conf.sum(axis=1), axis=0).mul(100).round(1)
    display(conf_pct)

    # Misclassified entries
    misclassified_multi = df_eval_multi[~df_eval_multi["correct"]].copy()
    print("Misclassified:", len(misclassified_multi), "/", len(df_eval_multi))

    cols = [
        "lemma", "ipa",
        "true_family", "pred_family",
        "origin_type", "primary_donor_family"
    ]

    if len(misclassified_multi) > 0:
        display(
            misclassified_multi[cols]
            .sample(min(sample_n, len(misclassified_multi)), random_state=random_state)
            .sort_values(["true_family", "pred_family", "lemma"])
        )
    else:
        print("No misclassified rows.")

    return {
        "confusion": conf,
        "confusion_pct": conf_pct,
        "misclassified": misclassified_multi,
    }


In [ ]:
all_sem_folds = multi_variant_models["+sem"]["summary"]["fold_evals"]
df_eval_multi_all = pd.concat(all_sem_folds, ignore_index=True)

inspect_multi_sem_all = inspect_multiclass_eval(
    df_eval_multi_all,
    sample_n=20,
)


# **Surface-form stress test with binary classification**


In [ ]:
import sklearn

model_bundle = binary_variant_models["+pos"]
clf = model_bundle["clf"]
vec = model_bundle["vec"]
summary = model_bundle["summary"]

print("classification file:", classification.__file__)
print("phonology file:", phonology.__file__)
print("sklearn version:", sklearn.__version__)
print("classes:", clf.classes_)
print("best_min_df:", model_bundle["best_min_df"])
print("n_features_in:", clf.n_features_in_)
print("vectorizer vocabulary:", len(vec.get_feature_names_out()))
print("macro_f1_mean for +pos binary:", summary["f1_macro_mean"])
print("log_loss_mean for +pos binary:", summary["log_loss_mean"])


In [ ]:

def analysis_surface(analysis: str) -> str:
    """Reconstruct the surface form covered by an emMorph analysis."""
    return "".join(surface for surface, tag in SEGMENT_RE.findall(analysis or ""))


def normalize_surface(s: str) -> str:
    return str(s).strip().lower()


def keep_exact_surface_analyses(analyses: list[dict], original_word: str) -> list[dict]:
    """Keep only analyses whose surface segments exactly match the input word."""
    target = normalize_surface(original_word)

    return [
        a for a in analyses
        if normalize_surface(analysis_surface(a.get("analysis", ""))) == target
    ]


def make_binary_true_label_map(df):

    loan_labels = {"loan", "international", "wanderword"}
    nonloan_labels = {"inherited", "internal", "onomatopoetic", "fictional"}

    def map_origin(origin):
        if pd.isna(origin):
            return pd.NA

        parts = {
            p.strip()
            for p in str(origin).split("|")
            if p.strip()
        }

        if parts and parts <= loan_labels:
            return "loan"

        if parts and parts <= nonloan_labels:
            return "non-loan"

        return pd.NA

    tmp = df.dropna(subset=["lemma"]).copy()
    tmp["true_label_tmp"] = tmp["origin_type"].apply(map_origin)

    label_map = {}

    for lemma, g in tmp.groupby("lemma", sort=False):
        labels = g["true_label_tmp"].dropna().unique()

        if len(labels) == 1:
            label_map[lemma] = labels[0]
        else:
            label_map[lemma] = pd.NA

    return label_map


# Words to test
# --> more word forms can be added

words = [
    "reform",
    "reformál",
    "megreformál",
    "megreformálás",
    "megreformálhat",
    "megreformálhatatlan",
    "megreformálhatatlanság",
    "archívál",
    "archívum",
    "archivál",
    "archiválás",
    "archiválhat",
    "archiválhatatlan",
    "archiválhatatlanság"
]

ueszweb_csv = pd.read_csv(INPUT_CSV)

print("Sampled words:", words)



with contextlib.redirect_stdout(io.StringIO()):
    analyses = emmorph_analyse_many(
        words,
        hfst_lookup=HFST_LOOKUP_PATH,
        hu_hfstol=HU_HFSTOL,
    )



# Keep only words with exact surface analyses from emmorph
# --> this is to test the classifier with additional suffixes

kept_words = []
skipped_words = []

for word in words:
    word_analyses = analyses.get(word, [])
    exact_analyses = keep_exact_surface_analyses(word_analyses, word)

    if exact_analyses:
        kept_words.append(word)
    else:
        skipped_words.append(word)
        print(f"No exact surface analysis for: {word}")

print("\nWords sent to classifier:")
print(kept_words)

if not kept_words:
    raise ValueError("No words had exact emMorph surface analyses.")


# binary +pos classifier
# --> we do not have the semantic categories of these words, so we do not use +sem 

model_bundle = binary_variant_models["+pos"]


clf = model_bundle["clf"]
vec = model_bundle["vec"]
cfg = model_bundle["cfg"]

print("\nClassifier classes:", getattr(clf, "classes_", None))

if len(getattr(clf, "classes_", [])) != 2:
    raise ValueError(
        "This is not a binary classifier. Re-run Cell 1 and make sure "
        "binary_variant_models['+pos'] is created from train_test_loan_classifier."
    )

lemma_to_ipa = phonemize_lemmas(
    kept_words,
    language="hu",
)

df_words = pd.DataFrame({
    "lemma": kept_words,
    "ipa": [
        lemma_to_ipa.get(str(word).strip().lower(), "")
        for word in kept_words
    ],
    # these are blank because we do not give the classifier gold etymology.
    "origin_type": "",
    "primary_donor_family": "",
    "pos": "",
    "first_year": "",
})

res = classification.score_binary_classifier_table(
    df_words,
    df_raw,
    clf,
    vec,
    ns=cfg["ns"],
    add_vowel_string=cfg["add_vowel_string"],
    add_vowel_patterns=cfg["add_vowel_patterns"],
    add_vowel_length_string=cfg.get("add_vowel_length_string", True),
    articulatory_backend=cfg["articulatory_backend"],
    semantic_feature=None,
    add_semantic_features=False,
    add_pos_features=cfg["add_pos_features"],
    include_missing_pos=cfg["include_missing_pos"],
    strip_final_ik=cfg["strip_final_ik"],
)


res["pred_label"] = res["pred_label"].replace({"native": "non-loan"})

true_label_map = make_binary_true_label_map(ueszweb_csv)

res["true_label"] = res["lemma"].map(true_label_map)
res["prediction_correct"] = (
    res["pred_label"] == res["true_label"]
).where(res["true_label"].notna(), pd.NA)

word_order = {w: i for i, w in enumerate(words)}
res = res.sort_values(
    "lemma",
    key=lambda s: s.map(word_order),
).reset_index(drop=True)


# Print results

print("\n--- Whole-surface-word predictions ---")
print(res[[
    "lemma",
    "ipa",
    "prob_loan",
    "pred_label",
    "true_label",
    "prediction_correct",
]].to_string(index=False))

if skipped_words:
    print("\nWords skipped because no exact emMorph surface analysis was found:")
    print(skipped_words)


In [ ]:
"""
Table 7.1:
The probabilities printed here differ from the values reported in the thesis because this notebook retrains the binary +pos classifier instead of loading the
exact fitted model used for the thesis table. Retraining from the current repository code and current Python/scikit-learn environment gives a different
set of learned coefficients, and therefore different loan probabilities.

The prediction pattern is the same: adding Hungarian derivational material lowers the loan probability for these surface forms, and the longest forms are
classified as non-loan-like.
""";
